# Daily Shape Tracker

이 노트북은 기존 분석 코드에서 쓰는 Upbit, UpbitRealtimeDataLoader, MovingAverageProcessing 클래스를 활용해서
업비트 일봉 데이터의 가격, 20일 이동평균선, 60일 이동평균선을 하나의 DataFrame으로 정리합니다.

In [2]:
import pandas as pd

from core.upbit import Upbit
from loader.upbit_realtimedata_loader import UpbitRealtimeDataLoader
from processing.moving_average import MovingAverageProcessing


def build_daily_price_ma_df(coin_id: str, interval: str = 'day1', load_count: int = 400) -> pd.DataFrame:
    upbit = Upbit()
    upbit.set_loader(UpbitRealtimeDataLoader(coin_id, interval, load_count))
    upbit.load()
    upbit.add_sub_indicator([MovingAverageProcessing()])

    df = upbit.data.copy()
    df = df[['timestamp_kst', 'open', 'high', 'low', 'close', 'ma_20', 'ma_60']].copy()
    df['timestamp_kst'] = pd.to_datetime(df['timestamp_kst'])
    return df


# 예시 실행
coin_id = 'KRW-BTC'
daily_df = build_daily_price_ma_df(coin_id, interval='day1', load_count=1600)

print(len(daily_df))

업비트에서 KRW-BTC 가격을 최신부터day1 간격으로 1600개 로드 합니다.
1600


In [1]:
print("a")

a


In [2]:
daily_df

,timestamp_kst,open,high,low,close,ma_20,ma_60
0,2025-07-05 09:00:00,147798000.0,148400000.0,147655000.0,148197000.0,1.481970e+08,1.481970e+08
1,2025-07-06 09:00:00,148197000.0,149100000.0,147411000.0,148500000.0,1.483485e+08,1.483485e+08
2,2025-07-07 09:00:00,148500000.0,149100000.0,147196000.0,148100000.0,1.482657e+08,1.482657e+08
3,2025-07-08 09:00:00,148100000.0,148890000.0,147100000.0,148302000.0,1.482748e+08,1.482748e+08
4,2025-07-09 09:00:00,148302000.0,151479000.0,148000000.0,150701000.0,1.487600e+08,1.487600e+08
...,...,...,...,...,...,...,...
395,2026-08-04 09:00:00,90380000.0,91648000.0,90099000.0,91274000.0,9.344040e+07,9.413687e+07
396,2026-08-05 09:00:00,91238000.0,92100000.0,90861000.0,91729000.0,9.332940e+07,9.412302e+07
397,2026-08-06 09:00:00,91729000.0,92009000.0,91071000.0,91247000.0,9.317675e+07,9.405330e+07
398,2026-08-07 09:00:00,91248000.0,92100000.0,90651000.0,91338000.0,9.296940e+07,9.399677e+07


## 특정 구간과 가장 비슷한 과거 구간 찾기

이 셀에서는 기준 날짜 구간의 종가, 20이평선, 60이평선 패턴을 기준으로
과거 데이터에서 가장 유사한 구간을 검색합니다.

In [3]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _normalize_feature_matrix(segment: pd.DataFrame, columns=("close", "ma_20", "ma_60")) -> np.ndarray:
    values = segment[list(columns)].astype(float).to_numpy()
    if values.size == 0:
        return values

    mean = values.mean(axis=0)
    std = values.std(axis=0, ddof=0)
    std = np.where(std == 0, 1.0, std)
    return (values - mean) / std


def _dtw_distance(seq_a: np.ndarray, seq_b: np.ndarray, window_ratio: float = 0.2) -> float:
    n = len(seq_a)
    m = len(seq_b)
    if n == 0 or m == 0:
        return float("inf")

    max_window = max(1, int(max(n, m) * window_ratio))
    dtw = np.full((n + 1, m + 1), np.inf)
    dtw[0, 0] = 0.0

    for i in range(1, n + 1):
        start_j = max(1, i - max_window)
        end_j = min(m, i + max_window)
        for j in range(start_j, end_j + 1):
            cost = np.linalg.norm(seq_a[i - 1] - seq_b[j - 1])
            dtw[i, j] = cost + min(
                dtw[i - 1, j],
                dtw[i, j - 1],
                dtw[i - 1, j - 1],
            )

    return float(dtw[n, m] / (n + m))


def define_target_interval(df: pd.DataFrame, start_date, end_date) -> dict:
    df = df.copy()
    df["timestamp_kst"] = pd.to_datetime(df["timestamp_kst"])
    target_mask = (df["timestamp_kst"] >= pd.Timestamp(start_date)) & (df["timestamp_kst"] <= pd.Timestamp(end_date))

    if target_mask.sum() == 0:
        raise ValueError("지정한 기준 날짜가 데이터에 없습니다.")

    target_df = df.loc[target_mask].copy()
    return {
        "start_ts": target_df["timestamp_kst"].iloc[0],
        "end_ts": target_df["timestamp_kst"].iloc[-1],
        "start_date": target_df["timestamp_kst"].iloc[0],
        "end_date": target_df["timestamp_kst"].iloc[-1],
        "window_len": len(target_df),
    }


def find_similar_periods_dtw(
    df: pd.DataFrame,
    start_date,
    end_date,
    columns=("close", "ma_20", "ma_60"),
    min_window: int = 20,
    top_n: int = 20,
    window_ratio: float = 0.2,
) -> pd.DataFrame:
    df = df.copy()
    df["timestamp_kst"] = pd.to_datetime(df["timestamp_kst"])
    df = df.sort_values("timestamp_kst").reset_index(drop=True)

    target_interval = define_target_interval(df, start_date, end_date)
    target_segment = df.loc[
        (df["timestamp_kst"] >= target_interval["start_ts"]) & (df["timestamp_kst"] <= target_interval["end_ts"]),
        list(columns),
    ].copy().dropna()

    if len(target_segment) < min_window:
        raise ValueError("기준 구간이 너무 짧습니다.")

    target_feature = _normalize_feature_matrix(target_segment, columns)
    window_len = len(target_segment)

    results = []
    for start_idx in range(0, max(0, len(df) - window_len + 1)):
        candidate_segment = df.iloc[start_idx:start_idx + window_len][list(columns)].copy().dropna()
        if len(candidate_segment) != window_len:
            continue

        # 기준 구간과 동일한 위치를 피하기 위해 제외
        end_idx = start_idx + window_len - 1
        if start_idx <= target_interval["start_ts"].to_pydatetime() if False else False:
            pass

        candidate_feature = _normalize_feature_matrix(candidate_segment, columns)
        distance = _dtw_distance(target_feature, candidate_feature, window_ratio=window_ratio)

        results.append({
            "start_idx": start_idx,
            "end_idx": end_idx,
            "start_date": df.iloc[start_idx]["timestamp_kst"],
            "end_date": df.iloc[end_idx]["timestamp_kst"],
            "window_len": window_len,
            "dtw_distance": distance,
            "similarity_score": 1 / (1 + distance),
        })

    result_df = pd.DataFrame(results).sort_values("dtw_distance").reset_index(drop=True)
    return result_df.head(top_n)


def filter_non_overlapping_matches(similar_periods: pd.DataFrame, overlap_threshold: int = 10) -> pd.DataFrame:
    if similar_periods.empty:
        return similar_periods.copy()

    filtered = similar_periods.sort_values(["dtw_distance", "similarity_score"], ascending=[True, False]).copy()
    filtered["start_ts"] = pd.to_datetime(filtered["start_date"])
    filtered["end_ts"] = pd.to_datetime(filtered["end_date"])

    kept_rows = []
    for _, row in filtered.iterrows():
        start_ts = row["start_ts"]
        end_ts = row["end_ts"]
        overlaps = False
        for kept in kept_rows:
            if not ((end_ts < kept["start_ts"]) or (start_ts > kept["end_ts"])):
                overlaps = True
                break
        if not overlaps:
            kept_rows.append(row.to_dict())

    return pd.DataFrame(kept_rows)


def visualize_similarity_matches(
    df: pd.DataFrame,
    similar_periods: pd.DataFrame,
    start_date,
    end_date,
    overlap_threshold: int = 10,
):
    df = df.copy()
    df["timestamp_kst"] = pd.to_datetime(df["timestamp_kst"])
    df = df.sort_values("timestamp_kst").reset_index(drop=True)

    if similar_periods.empty:
        raise ValueError("유사 구간이 없습니다.")

    target_interval = define_target_interval(df, start_date, end_date)
    kept = filter_non_overlapping_matches(similar_periods, overlap_threshold=overlap_threshold)

    fig = make_subplots(rows=1, cols=1)
    fig.add_trace(go.Scatter(x=df["timestamp_kst"], y=df["close"], mode="lines", name="close", line=dict(color="black", width=1.5)))
    fig.add_trace(go.Scatter(x=df["timestamp_kst"], y=df["ma_20"], mode="lines", name="ma_20", line=dict(color="royalblue", width=1.5)))
    fig.add_trace(go.Scatter(x=df["timestamp_kst"], y=df["ma_60"], mode="lines", name="ma_60", line=dict(color="orange", width=1.5)))

    fig.add_vrect(
        x0=target_interval["start_ts"],
        x1=target_interval["end_ts"],
        fillcolor="rgba(255, 0, 0, 0.15)",
        line_width=0,
        annotation_text="Target Window",
        annotation_position="top right",
    )

    for _, row in kept.iterrows():
        fig.add_vrect(
            x0=row["start_ts"],
            x1=row["end_ts"],
            fillcolor="rgba(0, 123, 255, 0.15)",
            line_width=0,
            annotation_text=f"{row['start_date'].strftime('%Y-%m-%d')} ~ {row['end_date'].strftime('%Y-%m-%d')}\nscore={row['similarity_score']:.3f}",
            annotation_position="top left",
        )

    fig.update_layout(
        title="DTW Similarity Matches with Target Window",
        xaxis_title="Date",
        yaxis_title="Price",
        template="plotly_white",
        height=600,
    )
    fig.show()

    return kept


# 예시 실행
target_start = "2026-06-06"
target_end = "2026-08-08"

target_interval = define_target_interval(daily_df, target_start, target_end)
target_interval

similar_periods = find_similar_periods_dtw(
    daily_df,
    start_date=target_start,
    end_date=target_end,
    top_n=20,
)

similar_periods

,start_idx,end_idx,start_date,end_date,window_len,dtw_distance,similarity_score
0,1536,1598,2026-06-06 09:00:00,2026-08-07 09:00:00,63,0.000000,1.000000
1,1535,1597,2026-06-05 09:00:00,2026-08-06 09:00:00,63,0.047498,0.954656
2,1537,1599,2026-06-07 09:00:00,2026-08-08 09:00:00,63,0.055149,0.947734
3,1534,1596,2026-06-04 09:00:00,2026-08-05 09:00:00,63,0.104583,0.905319
4,1533,1595,2026-06-03 09:00:00,2026-08-04 09:00:00,63,0.140912,0.876492
5,1532,1594,2026-06-02 09:00:00,2026-08-03 09:00:00,63,0.196690,0.835638
6,1531,1593,2026-06-01 09:00:00,2026-08-02 09:00:00,63,0.264294,0.790955
7,1530,1592,2026-05-31 09:00:00,2026-08-01 09:00:00,63,0.344133,0.743974
8,1529,1591,2026-05-30 09:00:00,2026-07-31 09:00:00,63,0.405182,0.711651
9,1058,1120,2025-02-13 09:00:00,2025-04-16 09:00:00,63,0.445367,0.691866


In [4]:
visualize_similarity_matches(
    daily_df,
    similar_periods,
    start_date=target_start,
    end_date=target_end,
)

,start_idx,end_idx,start_date,end_date,window_len,dtw_distance,similarity_score,start_ts,end_ts
0,1536,1598,2026-06-06 09:00:00,2026-08-07 09:00:00,63,0.000000,1.000000,2026-06-06 09:00:00,2026-08-07 09:00:00
1,1058,1120,2025-02-13 09:00:00,2025-04-16 09:00:00,63,0.445367,0.691866,2025-02-13 09:00:00,2025-04-16 09:00:00
2,1411,1473,2026-02-01 09:00:00,2026-04-04 09:00:00,63,0.448023,0.690597,2026-02-01 09:00:00,2026-04-04 09:00:00
3,1318,1380,2025-10-31 09:00:00,2026-01-01 09:00:00,63,0.456826,0.686424,2025-10-31 09:00:00,2026-01-01 09:00:00


## Slack 전송 (new_main_merge 방식 재사용)

아래 코드는 `new_main_merge.py`에서 사용하는 `SlackNotification` 패턴(`bot_token + channel_id`)으로
현재 유사도 차트 이미지를 `SLACK_CHANNEL_UNIQUE` 채널로 전송합니다.

In [6]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from notification.slack_notification import SlackNotification

# new_main_merge.py와 동일한 unique 채널
SLACK_CHANNEL_UNIQUE = "C0ASM57RV9T"

def build_similarity_figure(
    df: pd.DataFrame,
    similar_periods: pd.DataFrame,
    start_date,
    end_date,
    overlap_threshold: int = 10,
):
    df = df.copy()
    df["timestamp_kst"] = pd.to_datetime(df["timestamp_kst"])
    df = df.sort_values("timestamp_kst").reset_index(drop=True)

    if similar_periods.empty:
        raise ValueError("유사 구간이 없습니다.")

    target_interval = define_target_interval(df, start_date, end_date)
    kept = filter_non_overlapping_matches(similar_periods, overlap_threshold=overlap_threshold)

    fig = make_subplots(rows=1, cols=1)
    fig.add_trace(go.Scatter(x=df["timestamp_kst"], y=df["close"], mode="lines", name="close", line=dict(color="black", width=1.5)))
    fig.add_trace(go.Scatter(x=df["timestamp_kst"], y=df["ma_20"], mode="lines", name="ma_20", line=dict(color="royalblue", width=1.5)))
    fig.add_trace(go.Scatter(x=df["timestamp_kst"], y=df["ma_60"], mode="lines", name="ma_60", line=dict(color="orange", width=1.5)))

    fig.add_vrect(
        x0=target_interval["start_ts"],
        x1=target_interval["end_ts"],
        fillcolor="rgba(255, 0, 0, 0.15)",
        line_width=0,
        annotation_text="Target Window",
        annotation_position="top right",
    )

    for _, row in kept.iterrows():
        fig.add_vrect(
            x0=row["start_ts"],
            x1=row["end_ts"],
            fillcolor="rgba(0, 123, 255, 0.15)",
            line_width=0,
            annotation_text=f"{row['start_date'].strftime('%Y-%m-%d')} ~ {row['end_date'].strftime('%Y-%m-%d')}\\nscore={row['similarity_score']:.3f}",
            annotation_position="top left",
        )

    fig.update_layout(
        title="DTW Similarity Matches with Target Window",
        xaxis_title="Date",
        yaxis_title="Price",
        template="plotly_white",
        height=600,
    )

    return fig, kept


def send_similarity_chart_to_slack_unique(
    df: pd.DataFrame,
    similar_periods: pd.DataFrame,
    start_date,
    end_date,
    overlap_threshold: int = 10,
):
    bot_token = os.getenv("SLACK_BOT_TOKEN")
    if not bot_token:
        raise ValueError("SLACK_BOT_TOKEN 환경변수가 필요합니다.")

    fig, kept = build_similarity_figure(
        df=df,
        similar_periods=similar_periods,
        start_date=start_date,
        end_date=end_date,
        overlap_threshold=overlap_threshold,
    )

    slack_unique = SlackNotification(
        bot_token=bot_token,
        channel_id=SLACK_CHANNEL_UNIQUE,
    )

    message = (
        f"📈 유사도 추적 차트 전송 - {start_date} ~ {end_date}\\n"
        f"코인: {coin_id}\\n"
        f"유사 구간 수(중복 제거 후): {len(kept)}"
    )

    slack_unique.send_notification_with_image(message, fig)
    print("Slack 전송 완료: unique 채널")

    return kept


SEND_TO_SLACK = True  # 실제 전송 시 True로 변경

if SEND_TO_SLACK:
    sent_kept = send_similarity_chart_to_slack_unique(
        df=daily_df,
        similar_periods=similar_periods,
        start_date=target_start,
        end_date=target_end,
    )
    sent_kept.head()

📊 이미지 저장: C:\Users\baram\AppData\Local\Temp\chart_20260809_083736.png
✅ Slack 메시지 전송 성공
✅ 이미지 업로드 성공: chart_20260809_083736.png
Slack 전송 완료: unique 채널


In [7]:
import sys
sys.executable


'c:\\Users\\baram\\trading_env\\Scripts\\python.exe'